In [ ]:
!pip install roboflow ultralytics


In [ ]:
import cv2
from google.colab import files
from roboflow import Roboflow

rf = Roboflow(api_key="ZXx1HLukXKftna9VTt7n")
project = rf.workspace().project("air-pollution-vehicle-55jqd")
model = project.version(1).model

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

cap = cv2.VideoCapture(video_path)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter('output.mp4',
      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

vehicle_weights = {
    "two wheelers": 1,
    "three wheelers": 2,
    "passenger cars": 2,
    "light commercial vehicle": 3,
    "multi axle goods vehicles/multi axle buses": 4
}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, confidence=40).json()
    pollution_score = 0

    for pred in results["predictions"]:
        label = pred["class"].lower()
        pollution_score += vehicle_weights.get(label, 1)

        x, y, w, h = int(pred["x"]), int(pred["y"]), int(pred["width"]), int(pred["height"])
        cv2.rectangle(frame,
            (x - w//2, y - h//2),
            (x + w//2, y + h//2),
            (0, 255, 0), 2)
        cv2.putText(frame, pred["class"],
            (x - w//2, y - h//2 - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0), 1)

    # Percentage calculate
    pollution_percent = min((pollution_score / 50) * 100, 100)

    # Level decide
    if pollution_score == 0:
        level = "Clean"
        color = (0, 255, 0)
    elif pollution_score <= 5:
        level = "Low"
        color = (0, 255, 255)
    elif pollution_score <= 10:
        level = "Medium"
        color = (0, 165, 255)
    elif pollution_score <= 15:
        level = "High"
        color = (0, 0, 255)
    else:
        level = "Severe"
        color = (0, 0, 139)

    # Video-ல் காட்டு
    cv2.putText(frame, f"Pollution: {level}",
        (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)
    cv2.putText(frame, f"Level: {pollution_percent:.1f}%",
        (30, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

    out.write(frame)

cap.release()
out.release()
print("Done!")

# Convert & Download
import subprocess
subprocess.run(['ffmpeg', '-i', 'output.mp4',
               '-vcodec', 'libx264', 'final_output.mp4'])
files.download('final_output.mp4')

loading Roboflow workspace...
loading Roboflow project...


Saving air pollution.mp4 to air pollution.mp4
Done!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>